# Auto Loader

Auto Loader is new structured streaming source designed for large-scale, efficient data ingestion.
Auto Loader increamentally processes new data files as they arrive at cloud storage.

## Features
- efficient file detection using cloud services
- scaleablity improvements
- schema evolution and resiliency

#
### Read files using Auto Loader

In [0]:
from pyspark.sql.functions import col, current_timestamp

In [0]:
customer_df = spark.readStream \
  .format("cloudFiles") \
  .option("cloudFiles.format", "json") \
  .option("cloudFiles.schemaLocation", "/Volumes/gizmobox_catalog/landing/operational_data/customer_auto_loader/_schema") \
  .option("cloudFiles.inferColumnTypes", "true") \
  .option("cloudFiles.schemaHints", "date_of_birth date, member_since date, created_timestamp timestamp") \
  .load("/Volumes/gizmobox_catalog/landing/operational_data/customer_auto_loader/")

In [0]:
customer_transformed_df = customer_df \
    .withColumn("file_path", col("_metadata.file_path")) \
    .withColumn("ingestest_at", current_timestamp())

In [0]:
streaming_query = (
    customer_transformed_df
        .writeStream
        .format("delta")
        .option("checkpointLocation", "/Volumes/gizmobox_catalog/landing/operational_data/customer_auto_loader/_checkpoint")
        .toTable("gizmobox_catalog.bronze.customer_auto_loader")
)

In [0]:
%sql
select * from gizmobox_catalog.bronze.customer_auto_loader;

In [0]:
streaming_query.stop()

### Restart fresh
- delete all files in sink
- drop table

In [0]:
%sql
drop table gizmobox_catalog.bronze.customer_auto_loader;

In [0]:
dbutils.fs.rm("/Volumes/gizmobox_catalog/landing/operational_data/customer_auto_loader", recurse=True)

### File Options
- filter for 2024 files only

In [0]:
customer_df = spark.readStream \
  .format("cloudFiles") \
  .option("cloudFiles.format", "json") \
  .option("cloudFiles.schemaLocation", "/Volumes/gizmobox_catalog/landing/operational_data/customer_auto_loader/_schema") \
  .option("cloudFiles.inferColumnTypes", "true") \
  .option("cloudFiles.schemaHints", "date_of_birth date, member_since date, created_timestamp timestamp") \
  .option("pathGlobFilters", "customer_2024_*.json") \
  .load("/Volumes/gizmobox_catalog/landing/operational_data/customer_auto_loader/")

### Schema Evolution
- addNewColumns (default) -> stream fails -> but new columns are added to schema
- rescue -> stream continues -> schema is not evolved but new columns are recorded into rescued data column

In [0]:
customer_df = spark.readStream \
  .format("cloudFiles") \
  .option("cloudFiles.format", "json") \
  .option("cloudFiles.schemaLocation", "/Volumes/gizmobox_catalog/landing/operational_data/customer_auto_loader/_schema") \
  .option("cloudFiles.inferColumnTypes", "true") \
  .option("cloudFiles.schemaHints", "date_of_birth date, member_since date, created_timestamp timestamp") \
  .option("cloudFiles.schemaEvolutionMode", "rescue") \
  .load("/Volumes/gizmobox_catalog/landing/operational_data/customer_auto_loader/")